# 04 · Answering the question with the output table

**Question:** what is the typical price paid for a property in each municipality, each year, counting only sales the County codes as valid, of a single parcel?

This notebook reads only the gold table. It checks how much each year can be trusted, then looks at how prices moved, whether all places moved the same way, and what one municipality looks like row by row.

In [11]:
from helpers import connect, show

con, years = connect()
G = "gold.valid_sale_prices_by_municipality_year"
MIN_SALES = 30          # valid sales a row needs to be used in comparisons

show(f"""
SELECT count(*) AS rows, count(DISTINCT muni_code) AS municipalities,
       min(sale_year) AS first_year, max(sale_year) AS last_year,
       sum(records_total) AS records, sum(records_valid) AS valid_coded,
       sum(valid_records_excluded) AS valid_excluded, sum(valid_sales_used) AS valid_sales_used,
       count(median_price) AS rows_with_median, median(valid_sales_used) AS median_sales_per_row
FROM {G}
""")

┌───────┬────────────────┬────────────┬───────────┬─────────┬─────────────┬────────────────┬──────────────────┬──────────────────┬──────────────────────┐
│ rows  │ municipalities │ first_year │ last_year │ records │ valid_coded │ valid_excluded │ valid_sales_used │ rows_with_median │ median_sales_per_row │
│ int64 │     int64      │   int64    │   int64   │ int128  │   int128    │     int128     │      int128      │      int64       │        double        │
├───────┼────────────────┼────────────┼───────────┼─────────┼─────────────┼────────────────┼──────────────────┼──────────────────┼──────────────────────┤
│  2618 │            175 │       2012 │      2026 │  506977 │      114611 │            298 │           114313 │             2169 │                 19.0 │
└───────┴────────────────┴────────────┴───────────┴─────────┴─────────────┴────────────────┴──────────────────┴──────────────────┴──────────────────────┘



**What this shows:** 2,618 rows for 175 municipalities and Pittsburgh wards, 2012–2026. Together they cover all 506,977 records:
- 114,611 are coded valid
- 298 of those are left out (deed with several parcels or several dates, placeholder deed, no price, or the same parcel and date on two deeds)
- 114,313 are used for the prices

2,169 rows have a median; the other 449 have fewer than 5 valid sales. A typical row is based on 19 sales, which is too few to compare reliably. So for comparisons I only use rows with at least 30 sales (`MIN_SALES`).

## 1. How far can each year be trusted?

**Question:** before looking at any price: in each year, how many records are coded valid, how many haven't been reviewed by the County yet, and how many municipalities have enough sales to compare?

In [12]:
show(f"""
SELECT sale_year,
       sum(records_total) AS records,
       sum(valid_sales_used) AS valid_sales_used,
       round(100.0 * sum(records_valid) / sum(records_total), 1) AS pct_valid,
       round(100.0 * sum(records_undetermined) / sum(records_total), 1) AS pct_undetermined,
       count(*) FILTER (WHERE valid_sales_used >= {MIN_SALES}) AS municipalities_comparable,
       count(*) FILTER (WHERE median_price IS NULL) AS rows_without_median,
       bool_or(is_partial_year) AS partial, bool_or(is_provisional) AS provisional
FROM {G} GROUP BY 1 ORDER BY 1
""", rows=50)

# The latest year whose numbers are final, used as the end point below
LATEST = con.execute(f"SELECT max(sale_year) FROM {G} WHERE NOT is_provisional").fetchall()[0][0]
print("latest complete year:", LATEST)

┌───────────┬─────────┬──────────────────┬───────────┬──────────────────┬───────────────────────────┬─────────────────────┬─────────┬─────────────┐
│ sale_year │ records │ valid_sales_used │ pct_valid │ pct_undetermined │ municipalities_comparable │ rows_without_median │ partial │ provisional │
│   int64   │ int128  │      int128      │  double   │      double      │           int64           │        int64        │ boolean │   boolean   │
├───────────┼─────────┼──────────────────┼───────────┼──────────────────┼───────────────────────────┼─────────────────────┼─────────┼─────────────┤
│      2012 │   33651 │             2580 │       7.7 │             29.4 │                        26 │                  77 │ false   │ false       │
│      2013 │   34095 │             9093 │      27.0 │              2.6 │                        81 │                  23 │ false   │ false       │
│      2014 │   33670 │             5595 │      16.7 │              0.4 │                        65 │           

**What this shows:**
- **2012 is the weakest year.** Only 7.7% of its records are coded valid and 29.4% haven't been reviewed yet. Only 26 municipalities have enough sales, and 77 rows have no median.
- **2019 is the next weakest:** 12.8% valid, 53 municipalities with enough sales, 40 rows without a median. 2014, 2015 and 2018 are also below 20% valid.
- **From 2022 on** it's steady: about 30% valid, 80–89 municipalities with enough sales, and nothing left unreviewed until 2026.
- **2026 isn't finished** and its codes are still being reviewed, so the last complete year (`LATEST`) is **2025**.

**So:** how many records are coded valid depends on how the County coded each year, not on the market. That means the number of sales per year doesn't show market activity. 2012 and 2019 are based on few sales, and the 2012 level should be ignored (only 26 municipalities). I compare prices within each municipality, not as county totals.

## 2. How did the typical price move?

**Question:** what was the median of a typical municipality each year, and how much did it change from one year to the next?

The yearly change is measured inside each municipality: this year's median ÷ last year's, for municipalities with enough sales in both years. Then I take the median of those changes. This way, a different set of municipalities in one year can't create a change on its own.

In [13]:
show(f"""
WITH level AS (
    SELECT sale_year, count(*) AS municipalities, median(median_price) AS typical_median
    FROM {G} WHERE valid_sales_used >= {MIN_SALES} GROUP BY 1
), change AS (
    SELECT b.sale_year, count(*) AS municipalities_both_years,
           round(100 * (median(b.median_price * 1.0 / a.median_price) - 1), 1) AS typical_change_pct,
           round(100.0 * avg((b.median_price > a.median_price)::INT), 0) AS pct_municipalities_up
    FROM {G} a JOIN {G} b ON b.muni_code = a.muni_code AND b.sale_year = a.sale_year + 1
    WHERE a.valid_sales_used >= {MIN_SALES} AND b.valid_sales_used >= {MIN_SALES}
    GROUP BY 1
)
SELECT * FROM level LEFT JOIN change USING (sale_year) ORDER BY sale_year
""", rows=50)

# The year with the largest typical change, used to split the period below
JUMP = con.execute(f"""
    SELECT b.sale_year
    FROM {G} a JOIN {G} b ON b.muni_code = a.muni_code AND b.sale_year = a.sale_year + 1
    WHERE a.valid_sales_used >= {MIN_SALES} AND b.valid_sales_used >= {MIN_SALES} AND NOT b.is_provisional
    GROUP BY 1 ORDER BY median(b.median_price * 1.0 / a.median_price) DESC LIMIT 1
""").fetchall()[0][0]
print("year with the largest typical change:", JUMP)

┌───────────┬────────────────┬────────────────┬───────────────────────────┬────────────────────┬───────────────────────┐
│ sale_year │ municipalities │ typical_median │ municipalities_both_years │ typical_change_pct │ pct_municipalities_up │
│   int64   │     int64      │     double     │           int64           │       double       │        double         │
├───────────┼────────────────┼────────────────┼───────────────────────────┼────────────────────┼───────────────────────┤
│      2012 │             26 │       160625.0 │                      NULL │               NULL │                  NULL │
│      2013 │             81 │       132700.0 │                        26 │                7.8 │                  88.0 │
│      2014 │             65 │       140000.0 │                        65 │                0.6 │                  51.0 │
│      2015 │             63 │       144500.0 │                        60 │                6.4 │                  73.0 │
│      2016 │             73 │  

**What this shows:**
- **The level column depends on which municipalities are included.** 2012's typical median ($160,625) is higher than 2013's ($132,700) only because 2012 has 26 municipalities instead of 81. That's why I measure the change inside each municipality.
- **2013–2019:** the typical yearly change stays between −2.1% and +7.8%, and in 2018 and 2019 only about half of municipalities went up (50%, 46%).
- **2020 is the biggest change in the table: +24.1%, with 98% of municipalities up.** The query saves it as `JUMP`.
- **After that:** +11.6% in 2021, then +4% to +8% a year, with about three quarters of municipalities going up each year.

**So:** prices were roughly flat until 2019, jumped in 2020, and have grown steadily since. The jump isn't caused by coverage: 2020 has normal coverage (23.3% valid), and the change is measured in the same municipalities in both years.

## 3. Did all places move the same way?

**Question:** did cheap and expensive municipalities go up at the same rate? For each start year, I split municipalities into four groups (quarters) by their median that year, and measure how much each group grew up to the last complete year.

In [14]:
show(f"""
WITH pairs AS (
    SELECT a.sale_year AS start_year, a.muni_code, a.median_price AS start_median,
           b.median_price * 1.0 / a.median_price AS growth
    FROM {G} a JOIN {G} b ON b.muni_code = a.muni_code AND b.sale_year = {LATEST}
    WHERE a.sale_year < {LATEST} AND a.valid_sales_used >= {MIN_SALES} AND b.valid_sales_used >= {MIN_SALES}
), quartered AS (
    SELECT *, ntile(4) OVER (PARTITION BY start_year ORDER BY start_median) AS price_quarter FROM pairs
)
PIVOT (SELECT start_year, 'Q' || price_quarter AS q, growth FROM quartered)
ON q USING round(median(growth), 2)
ORDER BY start_year
""", rows=50)

┌────────────┬────────┬────────┬────────┬────────┐
│ start_year │   Q1   │   Q2   │   Q3   │   Q4   │
│   int64    │ double │ double │ double │ double │
├────────────┼────────┼────────┼────────┼────────┤
│       2012 │   2.32 │   2.07 │   2.01 │   1.84 │
│       2013 │   2.32 │    2.0 │   1.94 │   1.82 │
│       2014 │    2.3 │   1.96 │   1.93 │   1.83 │
│       2015 │   2.11 │   1.79 │   1.79 │   1.67 │
│       2016 │   2.21 │    1.8 │   1.84 │   1.57 │
│       2017 │   2.14 │   1.78 │   1.76 │   1.53 │
│       2018 │   2.21 │   1.77 │   1.69 │   1.56 │
│       2019 │   2.15 │   1.82 │   1.65 │   1.46 │
│       2020 │   1.56 │   1.38 │   1.38 │   1.31 │
│       2021 │   1.29 │   1.27 │    1.3 │   1.17 │
│       2022 │   1.19 │   1.14 │   1.16 │   1.12 │
│       2023 │   1.14 │   1.09 │   1.11 │   1.08 │
│       2024 │   1.06 │   1.04 │   1.03 │   1.02 │
└────────────┴────────┴────────┴────────┴────────┘
  13 rows                              5 columns



In [15]:
# The whole period split at the jump year: growth before and after, per municipality
show(f"""
WITH y AS (SELECT muni_code, sale_year, median_price AS p FROM {G} WHERE valid_sales_used >= {MIN_SALES}),
-- start: the first year with at least 50 comparable municipalities
first_ AS (SELECT min(sale_year) AS y FROM y WHERE sale_year < {JUMP} - 1
           AND sale_year IN (SELECT sale_year FROM y GROUP BY 1 HAVING count(*) >= 50))
SELECT (SELECT y FROM first_) AS from_year, {JUMP} - 1 AS split_year, {LATEST} AS to_year,
       count(*) AS municipalities,
       round(median(b.p * 1.0 / a.p), 2) AS typical_growth_before,
       round(median(c.p * 1.0 / b.p), 2) AS typical_growth_after,
       count(*) FILTER (WHERE c.p > b.p) AS rose_after
FROM y a JOIN y b USING (muni_code) JOIN y c USING (muni_code)
WHERE a.sale_year = (SELECT y FROM first_) AND b.sale_year = {JUMP} - 1 AND c.sale_year = {LATEST}
""")

┌───────────┬────────────┬─────────┬────────────────┬───────────────────────┬──────────────────────┬────────────┐
│ from_year │ split_year │ to_year │ municipalities │ typical_growth_before │ typical_growth_after │ rose_after │
│   int64   │   int32    │  int32  │     int64      │        double         │        double        │   int64    │
├───────────┼────────────┼─────────┼────────────────┼───────────────────────┼──────────────────────┼────────────┤
│      2013 │       2019 │    2025 │             53 │                  1.12 │                 1.72 │         53 │
└───────────┴────────────┴─────────┴────────────────┴───────────────────────┴──────────────────────┴────────────┘



**What this shows:**
- **Every start year gives the same order:** the cheapest quarter (Q1) grows the most up to 2025, and the most expensive (Q4) the least. From 2019 that's 2.15× vs 1.46×; from 2013, 2.32× vs 1.82×.
- **The gap opened before 2021.** From 2021 onward the four quarters grow almost the same (1.29 vs 1.17 from 2021, 1.06 vs 1.02 from 2024).
- **Before and after the jump:** I split at 2019 (the year before `JUMP`) and start at 2013, the first year with at least 50 municipalities to compare. The 53 municipalities with enough sales in all three years grew 1.12× from 2013 to 2019, and **1.72× from 2019 to 2025. All 53 went up.**

**So:** every municipality with enough sales to measure went up, and the cheapest ones went up the most, mostly during the 2020–2021 jump. *Assumption:* the mix of property types sold in each place stayed about the same; the table has no property type.

## 4. Do places keep their order, and how far apart are they?

**Question:** each year, how far apart are the cheapest and most expensive municipalities, and does their ranking change from one year to the next?

In [16]:
show(f"""
WITH ranked AS (
    SELECT sale_year, muni_code, median_price, rank() OVER (PARTITION BY sale_year ORDER BY median_price) AS r
    FROM {G} WHERE valid_sales_used >= {MIN_SALES}
), spread AS (
    SELECT sale_year, count(*) AS municipalities,
           quantile_cont(median_price, 0.1)::BIGINT AS p10_median, median(median_price)::BIGINT AS typical_median,
           quantile_cont(median_price, 0.9)::BIGINT AS p90_median,
           round(quantile_cont(median_price, 0.9) / quantile_cont(median_price, 0.1), 2) AS p90_to_p10
    FROM ranked GROUP BY 1
), order_kept AS (
    SELECT b.sale_year, round(corr(a.r, b.r), 2) AS rank_corr_with_previous_year
    FROM ranked a JOIN ranked b ON b.muni_code = a.muni_code AND b.sale_year = a.sale_year + 1
    GROUP BY 1
)
SELECT * FROM spread LEFT JOIN order_kept USING (sale_year) ORDER BY sale_year
""", rows=50)

┌───────────┬────────────────┬────────────┬────────────────┬────────────┬────────────┬──────────────────────────────┐
│ sale_year │ municipalities │ p10_median │ typical_median │ p90_median │ p90_to_p10 │ rank_corr_with_previous_year │
│   int64   │     int64      │   int64    │     int64      │   int64    │   double   │            double            │
├───────────┼────────────────┼────────────┼────────────────┼────────────┼────────────┼──────────────────────────────┤
│      2012 │             26 │      88850 │         160625 │     309098 │       3.48 │                         NULL │
│      2013 │             81 │      77450 │         132700 │     247500 │        3.2 │                         0.96 │
│      2014 │             65 │      75670 │         140000 │     275050 │       3.63 │                         0.95 │
│      2015 │             63 │      84960 │         144500 │     297000 │        3.5 │                         0.95 │
│      2016 │             73 │      80720 │         1420

In [17]:
# Highest and lowest in the latest complete year
show(f"""
(SELECT 'highest' AS end_, muni_desc, median_price, p25_price, p75_price, valid_sales_used FROM {G}
 WHERE sale_year = {LATEST} AND valid_sales_used >= {MIN_SALES} ORDER BY median_price DESC LIMIT 5)
UNION ALL
(SELECT 'lowest', muni_desc, median_price, p25_price, p75_price, valid_sales_used FROM {G}
 WHERE sale_year = {LATEST} AND valid_sales_used >= {MIN_SALES} ORDER BY median_price LIMIT 5)
""")

┌─────────┬────────────────────────┬──────────────┬───────────┬───────────┬──────────────────┐
│  end_   │       muni_desc        │ median_price │ p25_price │ p75_price │ valid_sales_used │
│ varchar │        varchar         │    int64     │   int64   │   int64   │      int64       │
├─────────┼────────────────────────┼──────────────┼───────────┼───────────┼──────────────────┤
│ highest │ Fox Chapel             │       977000 │    691814 │   1292500 │               48 │
│ highest │ Pine                   │       675000 │    446500 │    856250 │              132 │
│ highest │ Marshall               │       579125 │    415000 │    805000 │              116 │
│ highest │ Franklin Park          │       525000 │    365000 │    680000 │              157 │
│ highest │ 14th Ward - PITTSBURGH │       515000 │    342500 │    765500 │              263 │
│ lowest  │ Port Vue               │       123500 │    100000 │    144875 │               40 │
│ lowest  │ Stowe                  │       136750 

**What this shows:**
- **The ranking hardly moves.** Each year's ranking matches the year before very closely (rank correlation 0.95–0.98). Expensive places stay expensive and cheap places stay cheap.
- **The gap got smaller after the jump.** The municipality at the 90th percentile cost 3.2–4.0× the one at the 10th until 2019, 3.0–3.1× in 2020–2022, and 2.6–2.85× from 2023. The 10th percentile more than doubled ($80,605 in 2019 → $179,710 in 2025), while the 90th rose about 60% ($294,700 → $478,500).
- **Highest and lowest in 2025** (at least 30 sales): Fox Chapel $977,000 (48 sales), Pine $675,000 and Marshall $579,125 at the top; Port Vue $123,500, Stowe $136,750 and North Versailles $149,900 at the bottom. Pittsburgh wards show up at both ends (14th Ward $515,000, 29th Ward $155,000).

**So:** places move together and keep their order, but the gap between the cheapest and most expensive has been closing since 2020. This matches §3: the cheaper places rose faster.

## 5. The answer for one municipality

**Question:** what does the answer look like for one municipality, year by year? I use the municipality with the most valid sales over all years, so every year has a reliable median.

In [18]:
MUNI = con.execute(f"SELECT muni_code FROM {G} GROUP BY 1 ORDER BY sum(valid_sales_used) DESC LIMIT 1").fetchall()[0][0]
show(f"""
SELECT sale_year, muni_desc, median_price, p25_price, p75_price, valid_sales_used,
       pct_valid, pct_undetermined, is_low_sample, is_provisional
FROM {G} WHERE muni_code = '{MUNI}' ORDER BY sale_year
""", rows=50)

┌───────────┬────────────┬──────────────┬───────────┬───────────┬──────────────────┬───────────┬──────────────────┬───────────────┬────────────────┐
│ sale_year │ muni_desc  │ median_price │ p25_price │ p75_price │ valid_sales_used │ pct_valid │ pct_undetermined │ is_low_sample │ is_provisional │
│   int64   │  varchar   │    int64     │   int64   │   int64   │      int64       │  double   │      double      │    boolean    │    boolean     │
├───────────┼────────────┼──────────────┼───────────┼───────────┼──────────────────┼───────────┼──────────────────┼───────────────┼────────────────┤
│      2012 │ Penn Hills │        79450 │     65000 │     92250 │               60 │       5.6 │             34.8 │ false         │ false          │
│      2013 │ Penn Hills │        78000 │     62000 │     98500 │              247 │      20.4 │              1.9 │ false         │ false          │
│      2014 │ Penn Hills │        80000 │     63000 │    103500 │              193 │      15.7 │          

**What this shows:** Penn Hills has the most valid sales over all years, so every one of its rows is reliable (60 to 447 sales, never `is_low_sample`). Its typical sale stayed at $78,000–85,250 from 2012 to 2019, rose to $112,000 in 2020 and $135,000 in 2021, and reached $184,250 in 2025 (middle half $150,000–$215,000, 325 sales). That's 2.2× from 2019 to 2025, the same pattern as the cheaper quarter in §3. Its 2012 row also shows the coverage problem from §1: 5.6% valid and 34.8% not yet reviewed.

**So:** one row answers the question for one municipality and year, and the columns next to the price show how much it can be trusted.